# قِرْشَك | Qirshak — Smart Spending Watcher

مساعد مالي متعدد الوكلاء (CrewAI) يراقب صرفك، يكتشف الأنماط، وينبهك قبل ما تخسر السيطرة على ميزانيتك.

**بنية النوتبوك:**
1. الإعداد المشترك (الكل يشغّله أول)
2. المجموعة 1 — Categorizer + Pattern Detective
3. المجموعة 2 — Time-Based Alert + Predictor
4. المجموعة 3 — Budget & Savings Advisor + Tone Agent
5. تجميع الفريق الكامل (Crew) — يُكمّل بعد جهوزية كل المجموعات
6. التشغيل والاختبار النهائي

⚠️ كل عضو ينسخ القسم 1 كما هو بدون تعديل، ويشتغل على قسمه المخصص فقط.

# Section 1: Shared Setup

Everyone runs these four cells exactly as they are. Do not edit anything except your personal API key.

In [9]:
# 1. Install
!pip install -q -U crewai pandas

In [2]:
# 2. Imports
import os
import pandas as pd
from crewai import Agent, Task, Crew, Process, LLM

In [ ]:
# 3. Configure the LLM (each member enters their own key here at run time)
os.environ["OPENROUTER_API_KEY"] = ""

llm = LLM(
    model="openrouter/openai/gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)

In [5]:
# 4. Load the bank statement data (same file for everyone)

import urllib.request

url = "https://raw.githubusercontent.com/ramahIO/Qirshak/refs/heads/main/data/mock_statement.csv"
urllib.request.urlretrieve(url, "mock_statement.csv")

statement_df = pd.read_csv("mock_statement.csv")
statement_text = statement_df.to_string(index=False)

print("Statement loaded:", len(statement_df), "transactions")

Statement loaded: 43 transactions


In [14]:
# التحقق الآلي من المجاميع (Ground Truth) — يُستخدم لاحقاً كمرجع تحقق
category_map = {
    "STARBUCKS COFFEE JEDDAH": "coffee", "BARNS COFFEE": "coffee", "DR CAFE COFFEE": "coffee",
    "JAHEZ DELIVERY": "delivery", "HUNGERSTATION": "delivery",
    "NETFLIX.COM": "subscriptions", "SHAHID VIP": "subscriptions", "SPOTIFY PREMIUM": "subscriptions",
    "UBER TRIP": "transport", "CAREEM RIDE": "transport",
    "PANDA HYPERMARKET": "groceries", "CARREFOUR MARKET": "groceries",
    "RENT PAYMENT TRANSFER": "bills_rent", "STC MOBILE BILL": "bills_rent",
    "ELECTRICITY BILL SEC": "bills_rent", "STC PAY TRANSFER": "bills_rent",
    "SALARY - COMPANY XYZ": "income"
}

statement_df["category_verified"] = statement_df["description"].map(category_map)
true_totals = statement_df.groupby("category_verified")["amount_sar"].sum().round(2)

print("VERIFIED TOTALS (calculated directly from data, not by the LLM):")
print(true_totals)

VERIFIED TOTALS (calculated directly from data, not by the LLM):
category_verified
bills_rent      -1900.00
coffee           -435.50
delivery         -348.00
groceries        -973.50
income           4000.00
subscriptions     -88.98
transport         -90.50
Name: amount_sar, dtype: float64


In [15]:
print(statement_df.head(10))

         date   time              description  amount_sar category_verified
0  2026-07-01  09:00     SALARY - COMPANY XYZ     4000.00            income
1  2026-07-01  17:15  STARBUCKS COFFEE JEDDAH      -22.00            coffee
2  2026-07-02  08:30        PANDA HYPERMARKET     -186.50         groceries
3  2026-07-02  18:00           JAHEZ DELIVERY      -54.00          delivery
4  2026-07-03  11:00          STC MOBILE BILL     -120.00        bills_rent
5  2026-07-03  16:45             BARNS COFFEE      -19.00            coffee
6  2026-07-03  20:10                UBER TRIP      -28.50         transport
7  2026-07-04  17:30           DR CAFE COFFEE      -24.00            coffee
8  2026-07-05  09:00    RENT PAYMENT TRANSFER    -1500.00        bills_rent
9  2026-07-05  10:00              NETFLIX.COM      -39.99     subscriptions


---
# Section 2: Group 1 — Categorizer + Pattern Detective

**Receives:** `statement_text` directly from Section 1

**Delivers:** a text report with (a) every transaction categorized, (b) a list of detected patterns

In [16]:
categorizer = Agent(
    role="Personal Spending Categorizer",

    goal=(
        "Classify every transaction in a bank statement into one clear "
        "category, using only the transaction description and amount "
        "provided — never guessing beyond what the data shows."
    ),

    backstory=(
        "You are a meticulous financial analyst who reads raw bank "
        "statement rows and assigns each one to exactly one category: "
        "coffee, delivery, subscriptions, transport, groceries, "
        "bills_rent, income, or other. You never invent a transaction "
        "that isn't in the data, and you never assign two categories "
        "to the same row."
    ),

    llm=llm,
    verbose=True,
    allow_delegation=False
)

In [17]:
# --- Agent 2: Pattern Detective ---
pattern_detective = Agent(
    role="Spending Pattern Detective",

    goal=(
        "Analyze categorized spending data to surface behavioral patterns "
        "the person may not have noticed themselves — repetition, "
        "overlapping subscriptions, and timing trends — based only on "
        "the data provided."
    ),

    backstory=(
        "You are a behavioral finance analyst who specializes in finding "
        "quiet spending leaks: small repeated purchases that add up, "
        "subscriptions that overlap in purpose, and differences between "
        "weekday and weekend spending. You report only patterns that are "
        "clearly supported by the data — you never claim a subscription "
        "is 'unused' unless the data actually shows that, since usage "
        "isn't something a bank statement can confirm."
    ),

    llm=llm,
    verbose=True,
    allow_delegation=False
)

In [18]:
# --- Task 1: Categorization Task ---
categorization_task = Task(
    description=f'''
Categorize every transaction in the bank statement below into exactly
one of these categories: coffee, delivery, subscriptions, transport,
groceries, bills_rent, income, other.

{statement_text}

Rules:

- Use only the transaction description and amount provided.
- Do not invent transactions that are not in the data.
- Assign exactly one category per transaction — never two.
- Treat any positive amount as income.
- Group your output by category, and for each category state:
  the transactions in it, the count, and the total amount.
- Do not skip any transaction from the statement.
''',
    expected_output=(
        "A structured report grouping every transaction into its "
        "category, with a running total (SAR) and transaction count "
        "for each category."
    ),
    agent=categorizer
)

In [19]:
# --- Task 2: Pattern Detection Task ---
pattern_task = Task(
    description='''
Based on the categorized spending report above, identify behavioral
spending patterns.

Look specifically for:

1. Repeated small purchases in the same category (e.g. frequent coffee
   purchases) — note the frequency and typical time of day if visible
   in the data.
2. Subscriptions or recurring charges that fall into the same category
   (e.g. multiple streaming services) — flag them as worth reviewing,
   without claiming any of them are unused.
3. Any noticeable difference between spending on different days of the
   week, if the data shows one.

Rules:

- Base every pattern strictly on the categorized data — do not invent
  a pattern the data does not support.
- Do not make claims about whether a subscription is used or unused.
- If a pattern is weak or uncertain, say so explicitly rather than
  presenting it as a strong finding.
''',
    expected_output=(
        "A short list of clearly supported spending patterns, each with "
        "the evidence behind it (frequency, amount, or timing), and any "
        "patterns explicitly flagged as uncertain."
    ),
    agent=pattern_detective,
    context=[categorization_task]
)

In [20]:
# --- Test Group 1 on its own (mini Crew, no need to wait for other groups) ---
group1_crew = Crew(
    agents=[categorizer, pattern_detective],
    tasks=[categorization_task, pattern_task],
    process=Process.sequential,
    verbose=True
)

group1_result = await group1_crew.kickoff_async()
print(group1_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5b72873f-e082-43f6-83b1-7f57e80aa5f7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Categorize every transaction in the bank statement below into exactly                                          │
│  one of these categories: coffee, delivery, subscriptions, transport,                                           │
│  groceries, bills_rent, income, other.                                                                          │
│                                                                                                                 │
│        date  time             description  amount_sar                                                           │
│  2026-07-01 09:00    SALARY - COMPANY XYZ     4000.00                                                           │
│  2026-07-01 17:15 STARBUCKS COFFEE JEDDAH      -22.00                                                           │
│  2026-07-02 08:30       PANDA HYPERMARKET     -186.50                                                           │
│  2026-07-02 18:00          JAHEZ DELIVERY      -54.00                                                           │
│  2026-07-03 11:00         STC MOBILE BILL     -120.00                                                           │
│  2026-07-03 16:45            BARNS COFFEE      -19.00                                                           │
│  2026-07-03 20:10               UBER TRIP      -28.50                                                           │
│  2026-07-04 17:30          DR CAFE COFFEE      -24.00                                                           │
│  2026-07-05 09:00   RENT PAYMENT TRANSFER    -1500.00                                                           │
│  2026-07-05 10:00             NETFLIX.COM      -39.99                                                           │
│  2026-07-05 10:02              SHAHID VIP      -29.00                                                           │
│  2026-07-05 10:05         SPOTIFY PREMIUM      -19.99                                                           │
│  2026-07-05 16:50 STARBUCKS COFFEE JEDDAH      -21.00                                                           │
│  2026-07-06 12:20        CARREFOUR MARKET     -142.00                                                           │
│  2026-07-06 19:40           HUNGERSTATION      -63.00                                                           │
│  2026-07-07 17:10            BARNS COFFEE      -18.50                                                           │
│  2026-07-08 16:40 STARBUCKS COFFEE JEDDAH      -23.00                                                           │
│  2026-07-08 21:00             CAREEM RIDE      -32.00                                                           │
│  2026-07-09 17:55          DR CAFE COFFEE      -20.00                                                           │
│  2026-07-10 13:15       PANDA HYPERMARKET     -210.00                                                           │
│  2026-07-10 18:20          JAHEZ DELIVERY      -48.00                                                           │
│  2026-07-11 16:35 STARBUCKS COFFEE JEDDAH      -25.00                                                           │
│  2026-07-12 10:00    ELECTRICITY BILL SEC     -180.00                                                           │
│  2026-07-12 17:05            BARNS COFFEE      -19.50                                                           │
│  2026-07-13 09:45        STC PAY TRANSFER     -100.00                                                           │
│  2026-07-13 18:00           HUNGERSTATION      -58.00  

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Personal Spending Categorizer                                                                           │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Categorize every transaction in the bank statement below into exactly                                          │
│  one of these categories: coffee, delivery, subscriptions, transport,                                           │
│  groceries, bills_rent, income, other.                                                                          │
│                                                                                                                 │
│        date  time             description  amount_sar                                                           │
│  2026-07-01 09:00    SALARY - COMPANY XYZ     4000.00                                                           │
│  2026-07-01 17:15 STARBUCKS COFFEE JEDDAH      -22.00                                                           │
│  2026-07-02 08:30       PANDA HYPERMARKET     -186.50                                                           │
│  2026-07-02 18:00          JAHEZ DELIVERY      -54.00                                                           │
│  2026-07-03 11:00         STC MOBILE BILL     -120.00                                                           │
│  2026-07-03 16:45            BARNS COFFEE      -19.00                                                           │
│  2026-07-03 20:10               UBER TRIP      -28.50                                                           │
│  2026-07-04 17:30          DR CAFE COFFEE      -24.00                                                           │
│  2026-07-05 09:00   RENT PAYMENT TRANSFER    -1500.00                                                           │
│  2026-07-05 10:00             NETFLIX.COM      -39.99                                                           │
│  2026-07-05 10:02              SHAHID VIP      -29.00                                                           │
│  2026-07-05 10:05         SPOTIFY PREMIUM      -19.99                                                           │
│  2026-07-05 16:50 STARBUCKS COFFEE JEDDAH      -21.00                                                           │
│  2026-07-06 12:20        CARREFOUR MARKET     -142.00                                                           │
│  2026-07-06 19:40           HUNGERSTATION      -63.00                                                           │
│  2026-07-07 17:10            BARNS COFFEE      -18.50                                                           │
│  2026-07-08 16:40 STARBUCKS COFFEE JEDDAH      -23.00                                                           │
│  2026-07-08 21:00             CAREEM RIDE      -32.00                                                           │
│  2026-07-09 17:55          DR CAFE COFFEE      -20.00                                                           │
│  2026-07-10 13:15       PANDA HYPERMARKET     -210.00                                                           │
│  2026-07-10 18:20          JAHEZ DELIVERY      -48.00                                                           │
│  2026-07-11 16:35 STARBUCKS COFFEE JEDDAH      -25.00                                                           │
│  2026-07-12 10:00    ELECTRICITY BILL SEC     -180.00                                                           │
│  2026-07-12 17:05            BARNS COFFEE      -19.50                                                           │
│  2026-07-13 09:45        STC PAY TRANSFER     -100.00  

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Personal Spending Categorizer                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Coffee Transactions:**                                                                                       │
│  - STARBUCKS COFFEE JEDDAH: -22.00                                                                              │
│  - BARNS COFFEE: -19.00                                                                                         │
│  - DR CAFE COFFEE: -24.00                                                                                       │
│  - STARBUCKS COFFEE JEDDAH: -21.00                                                                              │
│  - STARBUCKS COFFEE JEDDAH: -25.00                                                                              │
│  - BARNS COFFEE: -18.50                                                                                         │
│  - STARBUCKS COFFEE JEDDAH: -23.00                                                                              │
│  - DR CAFE COFFEE: -20.00                                                                                       │
│  - STARBUCKS COFFEE JEDDAH: -24.00                                                                              │
│  - DR CAFE COFFEE: -22.50                                                                                       │
│  - STARBUCKS COFFEE JEDDAH: -24.00                                                                              │
│  - BARNS COFFEE: -19.50                                                                                         │
│  - STARBUCKS COFFEE JEDDAH: -26.00                                                                              │
│  - DR CAFE COFFEE: -21.00                                                                                       │
│  - STARBUCKS COFFEE JEDDAH: -22.50                                                                              │
│  - BARNS COFFEE: -19.00                                                                                         │
│  - STARBUCKS COFFEE JEDDAH: -23.50                                                                              │
│  - DR CAFE COFFEE: -20.50                                                                                       │
│  - STARBUCKS COFFEE JEDDAH: -24.50                                                                              │
│  - STARBUCKS COFFEE JEDDAH: -22.00                                                                              │
│                                                                                                                 │
│  **Count:** 20                                                                                                  │
│  **Total Amount:** -426.00 SAR                                                                                  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **Delivery Transactions:**                                                                                     │
│  - JAHEZ DELIVERY: -54.00                                                                                       │
│  - HUNGERSTATION: -63.00                               

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Spending Pattern Detective                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  1. **Repeated Small Purchases in Coffee Category:**                                                            │
│     - There are 20 transactions related to coffee, totaling -426.00 SAR.                                        │
│     - The frequency of purchases indicates a strong pattern of regular coffee consumption.                      │
│     - Notable purchases include multiple transactions at STARBUCKS COFFEE JEDDAH, with 10 occurrences, and DR   │
│  CAFE COFFEE, with 5 occurrences.                                                                               │
│     - The amounts spent range from -18.50 SAR to -26.00 SAR, suggesting a consistent spending behavior on       │
│  coffee.                                                                                                        │
│                                                                                                                 │
│  2. **Subscriptions or Recurring Charges:**                                                                     │
│     - There are 3 subscription transactions totaling -88.98 SAR:                                                │
│       - NETFLIX.COM: -39.99 SAR                                                                                 │
│       - SHAHID VIP: -29.00 SAR                                                                                  │
│       - SPOTIFY PREMIUM: -19.99 SAR                                                                             │
│     - These subscriptions fall into the entertainment category, and while they serve different purposes, they   │
│  may overlap in content consumption. This could be worth reviewing for potential redundancy.                    │
│                                                                                                                 │
│  3. **Transport Transactions:**                                                                                 │
│     - There are 3 transport-related transactions totaling -90.50 SAR:                                           │
│       - UBER TRIP: -28.50 SAR                                                                                   │
│       - CAREEM RIDE: -32.00 SAR                                                                                 │
│       - CAREEM RIDE: -30.00 SAR                                                                                 │
│     - This indicates a pattern of using ride-sharing services, but the frequency is low, making it a weaker     │
│  finding.                                                                                                       │
│                                                                                                                 │
│  4. **Groceries Spending:**                                                                                     │
│     - There are 6 grocery transactions totaling -1,073.50 SAR, with significant amounts spent at PANDA          │
│  HYPERMARKET and CARREFOUR MARKET.                                                                              │
│     - The amounts range from -95.00 SAR to -210.00 SAR, indicating a consistent pattern of grocery shopping,    │
│  but without specific timing data, the frequency of these purchases is less clear.                              │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Based on the categorized spending report above, identify behavioral                                            │
│  spending patterns.                                                                                             │
│                                                                                                                 │
│  Look specifically for:                                                                                         │
│                                                                                                                 │
│  1. Repeated small purchases in the same category (e.g. frequent coffee                                         │
│     purchases) — note the frequency and typical time of day if visible                                          │
│     in the data.                                                                                                │
│  2. Subscriptions or recurring charges that fall into the same category                                         │
│     (e.g. multiple streaming services) — flag them as worth reviewing,                                          │
│     without claiming any of them are unused.                                                                    │
│  3. Any noticeable difference between spending on different days of the                                         │
│     week, if the data shows one.                                                                                │
│                                                                                                                 │
│  Rules:                                                                                                         │
│                                                                                                                 │
│  - Base every pattern strictly on the categorized data — do not invent                                          │
│    a pattern the data does not support.                                                                         │
│  - Do not make claims about whether a subscription is used or unused.                                           │
│  - If a pattern is weak or uncertain, say so explicitly rather than                                             │
│    presenting it as a strong finding.                                                                           │
│                                                                                                                 │
│  Agent: Spending Pattern Detective                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

1. **Repeated Small Purchases in Coffee Category:**
   - There are 20 transactions related to coffee, totaling -426.00 SAR. 
   - The frequency of purchases indicates a strong pattern of regular coffee consumption. 
   - Notable purchases include multiple transactions at STARBUCKS COFFEE JEDDAH, with 10 occurrences, and DR CAFE COFFEE, with 5 occurrences. 
   - The amounts spent range from -18.50 SAR to -26.00 SAR, suggesting a consistent spending behavior on coffee.

2. **Subscriptions or Recurring Charges:**
   - There are 3 subscription transactions totaling -88.98 SAR:
     - NETFLIX.COM: -39.99 SAR
     - SHAHID VIP: -29.00 SAR
     - SPOTIFY PREMIUM: -19.99 SAR
   - These subscriptions fall into the entertainment category, and while they serve different purposes, they may overlap in content consumption. This could be worth reviewing for potential redundancy.

3. **Transport Transactions:**
   - There are 3 transport-related transactions totaling -90.50 SAR:
     - UBER TRIP: -28.

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 5b72873f-e082-43f6-83b1-7f57e80aa5f7                                                                       │
│  Final Output: 1. **Repeated Small Purchases in Coffee Category:**                                              │
│     - There are 20 transactions related to coffee, totaling -426.00 SAR.                                        │
│     - The frequency of purchases indicates a strong pattern of regular coffee consumption.                      │
│     - Notable purchases include multiple transactions at STARBUCKS COFFEE JEDDAH, with 10 occurrences, and DR   │
│  CAFE COFFEE, with 5 occurrences.                                                                               │
│     - The amounts spent range from -18.50 SAR to -26.00 SAR, suggesting a consistent spending behavior on       │
│  coffee.                                                                                                        │
│                                                                                                                 │
│  2. **Subscriptions or Recurring Charges:**                                                                     │
│     - There are 3 subscription transactions totaling -88.98 SAR:                                                │
│       - NETFLIX.COM: -39.99 SAR                                                                                 │
│       - SHAHID VIP: -29.00 SAR                                                                                  │
│       - SPOTIFY PREMIUM: -19.99 SAR                                                                             │
│     - These subscriptions fall into the entertainment category, and while they serve different purposes, they   │
│  may overlap in content consumption. This could be worth reviewing for potential redundancy.                    │
│                                                                                                                 │
│  3. **Transport Transactions:**                                                                                 │
│     - There are 3 transport-related transactions totaling -90.50 SAR:                                           │
│       - UBER TRIP: -28.50 SAR                                                                                   │
│       - CAREEM RIDE: -32.00 SAR                                                                                 │
│       - CAREEM RIDE: -30.00 SAR                                                                                 │
│     - This indicates a pattern of using ride-sharing services, but the frequency is low, making it a weaker     │
│  finding.                                                                                                       │
│                                                                                                                 │
│  4. **Groceries Spending:**                                                                                     │
│     - There are 6 grocery transactions totaling -1,073.50 SAR, with significant amounts spent at PANDA          │
│  HYPERMARKET and CARREFOUR MARKET.                                                                              │
│     - The amounts range from -95.00 SAR to -210.00 SAR, indicating a consistent pattern of grocery shopping,    │
│  but without specific timing data, the frequency of these purchases is less clear.                              │
│                                                       

---
# Section 3: Group 2 — Time-Based Alert + Predictor

**Receives:** Group 1's output (or a temporary placeholder for early independent testing)

**Delivers:** (a) a live alert if triggered, (b) a projection of when the balance will run out before month end

In [ ]:
# Temporary placeholder representing the shape of Group 1's output
# (replaced later with context=[categorization_task, pattern_task] at merge time)
placeholder_categorized_data = """
Categories found: Coffee (15 transactions, avg 22 SAR, mostly between 16:00-19:00),
Subscriptions (Netflix, Shahid, Spotify - all charged same day, same category),
Groceries (4 transactions), Rent (1500 SAR), Bills (300 SAR total),
Transport (3 transactions), Income: 4000 SAR salary on day 1.
"""

In [ ]:
# --- Agent 3: Time-Based Alert Agent ---
time_alert_agent = Agent(
    role="TODO",
    goal="TODO",
    backstory="TODO",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

In [ ]:
# --- Agent 4: Predictor ---
predictor = Agent(
    role="TODO",
    goal="TODO",
    backstory="TODO",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

In [ ]:
# --- Task 3: Time-Based Alert Task ---
alert_task = Task(
    description=f'''
TODO: Based on the time pattern below, decide whether the current (simulated)
time warrants an alert:

{placeholder_categorized_data}

TODO: add a "current simulated time" variable and a confidence threshold rule
''',
    expected_output="TODO",
    agent=time_alert_agent
)

In [ ]:
# --- Task 4: Prediction Task ---
prediction_task = Task(
    description=f'''
TODO: Based on the current spending rate, project whether the balance will
run out before month end, and by how many days:

{placeholder_categorized_data}
''',
    expected_output="TODO",
    agent=predictor
)

In [ ]:
# --- Test Group 2 on its own ---
group2_crew = Crew(
    agents=[time_alert_agent, predictor],
    tasks=[alert_task, prediction_task],
    process=Process.sequential,
    verbose=True
)

group2_result = await group2_crew.kickoff_async()
print(group2_result.raw)

---
# Section 4: Group 3 — Budget & Savings Advisor + Tone Agent

**Receives:** Groups 1 and 2's output (or a temporary placeholder for early independent testing)

**Delivers:** the final report (budget plan) in Qirshak's friendly tone

In [ ]:
# Temporary placeholder representing the shape of Groups 1 and 2's output
placeholder_full_analysis = """
Categorized spending: Coffee 340 SAR/month, Subscriptions 89 SAR/month,
Groceries 800 SAR/month, Rent 1500 SAR, Bills 300 SAR, Transport 120 SAR.
Prediction: at current spending rate, balance will run out approximately
3-5 days before month end. Income: 4000 SAR.
"""

In [ ]:
# --- Agent 5: Budget & Savings Advisor ---
budget_advisor = Agent(
    role="TODO",
    goal="TODO",
    backstory="TODO",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

In [ ]:
# --- Agent 6: Tone Agent ---
tone_agent = Agent(
    role="TODO",
    goal="TODO — rewrites tone only, never changes any number or fact",
    backstory="TODO",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

In [ ]:
# --- Task 5: Budget & Savings Task ---
budget_task = Task(
    description=f'''
TODO: Based on the analysis below, build a realistic daily spending limit
per category and a monthly savings target:

{placeholder_full_analysis}
''',
    expected_output="TODO",
    agent=budget_advisor
)

In [ ]:
# --- Task 6: Tone Rewrite Task ---
tone_task = Task(
    description='''
TODO: Rewrite the report above in Qirshak's warm, lightly humorous Arabic
voice, without changing any number or fact stated in it.
''',
    expected_output="TODO",
    agent=tone_agent,
    context=[budget_task]
)

In [ ]:
# --- Test Group 3 on its own ---
group3_crew = Crew(
    agents=[budget_advisor, tone_agent],
    tasks=[budget_task, tone_task],
    process=Process.sequential,
    verbose=True
)

group3_result = await group3_crew.kickoff_async()
print(group3_result.raw)

---
# Section 5: Assemble the Full Crew — Integration Owner Only

**Do not run these cells until every group's agents work successfully on their own.**

Merge steps:
1. Delete both placeholders (`placeholder_categorized_data`, `placeholder_full_analysis`)
2. Update `context=[...]` on each Task to link the real tasks together (not the placeholder text)
3. Order the six agents and tasks in the correct dependency order

In [ ]:
# TODO (Integration Owner):
# 1. Update alert_task and prediction_task to use:
#    context=[categorization_task, pattern_task]
#    instead of placeholder_categorized_data
#
# 2. Update budget_task to use:
#    context=[pattern_task, alert_task, prediction_task]
#    instead of placeholder_full_analysis

qirshak_crew = Crew(
    agents=[
        categorizer,
        pattern_detective,
        time_alert_agent,
        predictor,
        budget_advisor,
        tone_agent
    ],
    tasks=[
        categorization_task,
        pattern_task,
        alert_task,
        prediction_task,
        budget_task,
        tone_task
    ],
    process=Process.sequential,
    verbose=True
)

print("Qirshak full crew (6 agents) assembled successfully.")

---
# Section 6: Final Run & Test

In [ ]:
final_result = await qirshak_crew.kickoff_async()
print(final_result.raw)

In [ ]:
# Display each agent's output separately (for review and the live demo)
labels = [
    "CATEGORIZER", "PATTERN DETECTIVE", "TIME-BASED ALERT",
    "PREDICTOR", "BUDGET & SAVINGS ADVISOR", "TONE AGENT (FINAL REPORT)"
]

for label, output in zip(labels, final_result.tasks_output):
    print("\n" + "=" * 60)
    print(label)
    print("=" * 60)
    print(output.raw)